# Probabilistic Graphical Model Learning 

In [2]:
%load_ext autotime
%load_ext nb_black
%matplotlib inline

import os
from collections import defaultdict
import torch
import numpy as np
import pandas as pd
import cvxpy as cp
import scipy.stats
from scipy import optimize
from sympy import *
from torch.distributions import constraints
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 300
plt.rcParams["figure.figsize"] = (16, 12)

import pyro
import pyro.distributions as dist
from pyro import poutine
from pyro.infer.autoguide import AutoDelta
from pyro.optim import Adam
from pyro.infer import SVI, TraceEnum_ELBO, config_enumerate, infer_discrete

smoke_test = "CI" in os.environ
assert pyro.__version__.startswith("1.5.1")
pyro.enable_validation(True)

The autotime extension is already loaded. To reload it, use:
  %reload_ext autotime
The nb_black extension is already loaded. To reload it, use:
  %reload_ext nb_black
time: 527 ms (started: 2021-02-03 15:55:15 +08:00)


<IPython.core.display.Javascript object>

In [3]:
pd.read_csv("../../data/external/WA_Fn-UseC_-Telco-Customer-Churn.csv")

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5,No
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.6,Yes


time: 83.7 ms (started: 2021-02-03 15:55:31 +08:00)


<IPython.core.display.Javascript object>

## Problem Description

Given the statistical model which generates a set $\mathbf {x}$  of observed data, a set of unobserved latent data or missing values $\mathbf {z}$ , and a vector of unknown parameters $\boldsymbol {\theta }$, along with a likelihood function $L({\boldsymbol {\theta }};\mathbf {x} ,\mathbf {z} )=p(\mathbf {x} ,\mathbf {z} \mid {\boldsymbol {\theta }})$, we have 2 tasks that are of interest:

- 1. Find the MLE ($\boldsymbol {\theta}$ is parameter) / MAP ($\boldsymbol {\theta}$ is random variable) estimates of the model parameters $\boldsymbol {\theta_{\rm{max}}}$ {cite}`Expectat45:online`:

\begin{aligned}
    \boldsymbol {\theta_{\rm{max}}} &= \underset{\boldsymbol {\theta}}{\operatorname{argmax}} \log p(\mathbf{x} \mid \boldsymbol{\theta }) \\
    &= \underset{\boldsymbol {\theta}}{\operatorname{argmax}} \log \int p(\mathbf {x} ,\mathbf {z} \mid {\boldsymbol {\theta }})\,d\mathbf {z} \\
\end{aligned}

where $p(\mathbf{x} \mid \boldsymbol{\theta })$ is known as the ***Evidence / Marginal Likelihood / Incomplete-data Likelihood*** and $p(\mathbf {x} ,\mathbf {z} \mid {\boldsymbol {\theta }})$ is known as the ***Complete-data likelihood***.

- 2. Find the posterior over the latent variables $\mathbf{z}$, $p(\mathbf {z} \mid \mathbf {x}, \boldsymbol {\theta_{\rm{max}} })$ {cite}`SVIPartI61:online`:

\begin{aligned}
    p(\mathbf {z} \mid \mathbf {x}, \boldsymbol {\theta_{\rm{max}} }) &= \frac{p(\mathbf {x}, \mathbf {z} \mid \boldsymbol {\theta_{\rm{max}} })}{
\int p(\mathbf {x}, \mathbf {z} \mid \boldsymbol {\theta_{\rm{max}} })\,d\mathbf{z}} \\
\end{aligned}

However, this $p(\mathbf{x} \mid \boldsymbol{\theta }) = \int p(\mathbf {x} ,\mathbf {z} \mid {\boldsymbol {\theta }})\,d\mathbf {z}$ is often intractable (e.g. if $\mathbf {z}$  is a sequence of events, so that the number of values grows exponentially with the sequence length, the exact calculation of the sum will be extremely difficult). Let's instead try to find a lower bound for it by expanding it {cite}`SVIPartI61:online`:

\begin{aligned}
    \log p(\mathbf{x} \mid \boldsymbol{\theta }) 
    &= \log \int p(\mathbf {x} ,\mathbf {z} \mid {\boldsymbol {\theta }})\,d\mathbf {z} \\
\end{aligned}

Using ideas from importance sampling, assume we have another variational distribution [approximate posterior distribution to $p({\bf z} \mid {\bf x}, {\bf \theta})$)], $q({\bf z} \mid {\bf \phi})$, where $q({\bf z} \mid {\bf \phi}) > 0$ whenever $p({\bf z}) = \int p({\bf x}, {\bf z} \mid {\bf \theta}) d{ \bf x} > 0$, and we rewite:

\begin{aligned}
    \log p(\mathbf{x} \mid \boldsymbol{\theta }) 
    &= \log \int p(\mathbf {x} ,\mathbf {z} \mid {\boldsymbol {\theta }}) \frac{q({\bf z} \mid {\bf \phi})}{q({\bf z} \mid {\bf \phi})} \,d\mathbf {z} \\
    &= \log \operatorname {E}_{q({\bf z} \mid {\bf \phi})} \left[\frac{p(\mathbf {x} ,\mathbf {z} \mid {\boldsymbol {\theta }})}{q({\bf z} \mid {\bf \phi})} \right] \\
\end{aligned}

By Jensen's Inequality, given concave function $f(X)$ (e.g. $\log$), $f\operatorname {E}\left[X\right] \geq \operatorname {E}\left[f(X)\right]$ {cite}`Variatio28:online`:

\begin{aligned}
    \log p(\mathbf{x} \mid \boldsymbol{\theta }) 
    &= \log \operatorname {E}_{q({\bf z} \mid {\bf \phi})} \left[\frac{p(\mathbf {x} ,\mathbf {z} \mid {\boldsymbol {\theta }})}{q({\bf z} \mid {\bf \phi})} \right] \\
    &\geq \operatorname {E}_{q({\bf z} \mid {\bf \phi})} \left[\log\left(\frac{p(\mathbf {x} ,\mathbf {z} \mid {\boldsymbol {\theta }})}{q({\bf z} \mid {\bf \phi})}\right)\right] \\
    &= \operatorname {E}_{q({\bf z} \mid {\bf \phi})} \left[\log p(\mathbf {x} ,\mathbf {z} \mid {\boldsymbol {\theta }}) - \log q({\bf z} \mid {\bf \phi})\right] \\
    &= \operatorname {E}_{q({\bf z} \mid {\bf \phi})} \left[\log p(\mathbf {x} ,\mathbf {z} \mid {\boldsymbol {\theta }})\right] - \operatorname {E}_{q({\bf z} \mid {\bf \phi})} \left[\log q({\bf z} \mid {\bf \phi})\right] \\
    &= \underbrace{\underbrace{\operatorname {E}_{q({\bf z} \mid {\bf \phi})} \left[\log p(\mathbf {x} ,\mathbf {z} \mid {\boldsymbol {\theta }})\right]}_{\text{Expected Complete-data Log Likelihood}} + \underbrace{\operatorname{H}\left[\log q({\bf z} \mid {\bf \phi})\right]}_{\text{Entropy of Variational Dist.}}}_{\text{ELBO / Negative Variational Free Energy } \mathcal{L}(q({\bf z}\mid {\bf \phi}))} \\
\end{aligned}

Hence, we get an ***Evidence Lower Bound (ELBO)*** (also known as the ***Negative Variational Free Energy***) on the $\log$ Evidence. Instead of an inequality, we can get an exact equality of the form below by deriving the ELBO from rearranging the KL Divergence from our variational distribution (approximate posterior of latent variables) $q({\bf z} \mid {\bf \phi})$ to our actual posterior over latent variables $p({\bf z} \mid {\bf x}, {\bf \theta})$:

Derivation from ${\rm KL}(q({\bf z} \mid {\bf \phi}) \mid\mid p({\bf z} \mid {\bf x}, {\bf \theta}))$:

\begin{aligned}
    {\rm KL}(q({\bf z} \mid {\bf \phi}) \mid\mid p({\bf z} \mid {\bf x}, {\bf \theta}))
    &= \operatorname{E}_{q({\bf z} \mid {\bf \phi})}\left[\log\left(\frac{q({\bf z} \mid {\bf \phi})}{p({\bf z} \mid {\bf x}, {\bf \theta})}\right)\right] \\
    &= \operatorname{E}_{q({\bf z} \mid {\bf \phi})}\left[\log q({\bf z} \mid {\bf \phi})\right] - \operatorname{E}_{q({\bf z} \mid {\bf \phi})}\left[\log p({\bf z} \mid {\bf x}, {\bf \theta})\right] \\
    &= \operatorname{E}_{q({\bf z} \mid {\bf \phi})}\left[\log q({\bf z} \mid {\bf \phi})\right] - \operatorname{E}_{q({\bf z} \mid {\bf \phi})}\left[\log p({\bf z}, {\bf x} \mid {\bf \theta})\right] + \operatorname{E}_{q({\bf z} \mid {\bf \phi})}\left[\log p({\bf x} \mid {\bf \theta})\right] \\
    &= -\left[\operatorname {E}_{q({\bf z} \mid {\bf \phi})} \left[\log p(\mathbf {x} ,\mathbf {z} \mid {\boldsymbol {\theta }})\right] + \operatorname{H}\left[\log q({\bf z} \mid {\bf \phi})\right]\right] + \operatorname{E}_{q({\bf z} \mid {\bf \phi})}\left[\log p({\bf x} \mid {\bf \theta})\right] \\
    &= -\mathcal{L}(q({\bf z}\mid {\bf \phi})) + \log p({\bf x} \mid {\bf \theta}) \because \text{Expectation is over latent variables }{\bf z}\text{, which is independent of }{\bf x} \\
\end{aligned}

\begin{aligned}
    \therefore \log p({\bf x} \mid {\bf \theta}) 
    &= \mathcal{L}(q({\bf z} \mid {\bf \phi})) + {\rm KL}(q({\bf z} \mid {\bf \phi}) \mid\mid p({\bf z} \mid {\bf x}, {\bf \theta})) \\
\end{aligned}

Since $\log p({\bf x} \mid {\bf \theta})$ is a constant, maximizing our ELBO / Negative Variational Free Energy will be equivalent to minimizing the ${\rm KL}(q({\bf z} \mid {\bf \phi}) \mid\mid p({\bf z} \mid {\bf x}, {\bf \theta}))$ (0 when $q({\bf z} \mid {\bf \phi}) = p({\bf z} \mid {\bf x}, {\bf \theta})$), making our variational approximation as close as possible to the actual posterior over latents. After this procedure, our 2 tasks will look like:

- 1. Find the MLE (${\bf\theta}, {\bf\phi}$ are parameters) / MAP (${\bf\theta}, {\bf\phi}$ are random variables) estimates of the model parameters ${\bf \theta_{\rm{max}}}, {\bf \phi_{\rm{max}}}$ by maximizing the ELBO:

\begin{aligned}
    {\bf\theta_{\rm{max}}} &= \underset{\boldsymbol {\theta}}{\operatorname{argmax}} \log p(\mathbf{x} \mid \boldsymbol{\theta }) \\
    {\bf\theta_{\rm{max}}}, {\bf\phi_{\rm{max}}} &\approx \underset{{\bf \theta}, {\bf \phi}}{\operatorname{argmax}} \mathcal{L}(q({\bf z} \mid {\bf \phi})) \\
    &= \underset{{\bf \theta}, {\bf \phi}}{\operatorname{argmax}} \operatorname {E}_{q({\bf z} \mid {\bf \phi})} \left[\log p(\mathbf {x} ,\mathbf {z} \mid {\boldsymbol {\theta }})\right] - \operatorname{H}\left[\log q({\bf z} \mid {\bf \phi})\right]  \\
\end{aligned}

In maximizing the ELBO, the first term, Expected Complete-data Log Likelihood, encourages the MLE / MAP estimates of the model parameters to be 

- 2. Find the posterior over the latent variables $\mathbf{z}$, $p(\mathbf {z} \mid \mathbf {x}, \boldsymbol {\theta_{\rm{max}} })$ {cite}`SVIPartI61:online`:

\begin{aligned}
    p(\mathbf {z} \mid \mathbf {x}, \boldsymbol {\theta_{\rm{max}} }) &\approx q({\bf z} \mid {\bf \phi}) \\
\end{aligned}

### Technique 1: Expectation-Maximization

The EM algorithm seeks to find the MLE of the evidence / marginal likelihood / incomplete-data likelihood by iteratively applying these two steps {cite}`Expectat45:online`:

- 1. Expectation step (E step): Set the approximate posterior / variational distribution $q({\bf z}\mid {\bf \phi}) = p(\mathbf {z} \mid \mathbf {x}, \boldsymbol {\theta_{t} })$, where $\bf \theta_{t}$ are the previous M-step estimates of $\bf \theta$, this way the ${\rm KL}(q({\bf z} \mid {\bf \phi}) \mid\mid p({\bf z} \mid {\bf x}, {\bf \theta})) = 0$ and $\log p({\bf x} \mid {\bf \theta}) = \mathcal{L}(p({\bf z} \mid {\bf x}, {\bf \theta_{t}}))$. Our objective is then to 

    - A. Calculate the posterior over latent variables $p(\mathbf {z} \mid \mathbf {x} ,{\boldsymbol {\theta }}^{(t)})$ and 
    
    - B. Calculate $Q({\boldsymbol {\theta }}\mid {\boldsymbol {\theta }}^{(t)})$:

\begin{aligned}
    Q({\boldsymbol {\theta }}\mid {\boldsymbol {\theta }}^{(t)}) &= \operatorname {E} _{p(\mathbf {z} \mid \mathbf {x} ,{\boldsymbol {\theta }}^{(t)})}\left[\log L({\boldsymbol {\theta }};\mathbf {x} ,\mathbf {z} )\right]\, \\
    &= \operatorname {E} _{p(\mathbf {z} \mid \mathbf {x} ,{\boldsymbol {\theta }}^{(t)})}\left[\log p(\mathbf {x} ,\mathbf {z} \mid {\boldsymbol {\theta }}) \right]\, \\
    &= \int p(\mathbf {z} \mid \mathbf {x} ,{\boldsymbol {\theta }}^{(t)}) \log p(\mathbf {x} ,\mathbf {z} \mid {\boldsymbol {\theta }})\,d\mathbf{z} \\
\end{aligned}

Notice that the only thing that is missing from $Q({\boldsymbol {\theta }}\mid {\boldsymbol {\theta }}^{(t)})$ compared to the ELBO is the entropy of the approximate posterior distribution $\operatorname{H}\left[\log q({\bf z} \mid {\bf \phi})\right]$.

- 2. Maximization step (M step): Find the parameters that maximize $ Q({\boldsymbol {\theta }}\mid {\boldsymbol {\theta }}^{(t)})$:

\begin{aligned} 
    {\boldsymbol {\theta }}^{(t+1)} &= {\underset {\boldsymbol {\theta }}{\operatorname {arg\,max} }}\ Q({\boldsymbol {\theta }}\mid {\boldsymbol {\theta }}^{(t)})\,
\end{aligned}

#### Example 01 - Gaussian Mixture Model

#### Example 02 - Latent Dirichlet Allocation Topic Model

#### Example 03 - Hidden Markov Model

### Technique 2: Markov Chain Monte Carlo

#### Example 01 - Gaussian Mixture Model

#### Example 02 - Latent Dirichlet Allocation Topic Model

#### Example 03 - Hidden Markov Model

#### Example 04 - Bayesian Linear Regression

### Technique 3: Variational Inference

\begin{aligned}
    {\rm ELBO} &\equiv \mathbb{E}_{q({\bf z} \mid {\bf \phi})} \left [
\log p({\bf x}, {\bf z} \mid {\bf \theta}) - \log q({\bf z} \mid {\bf \phi})
\right]
\end{aligned}

#### a. Mean-field Approximation 

#### Example 01 - Gaussian Mixture Model

#### Example 02 - Latent Dirichlet Allocation Topic Model

#### Example 03 - Hidden Markov Model

#### b. Black-box Variational Inference

#### Example 01 - Gaussian Mixture Model

#### Example 02 - Latent Dirichlet Allocation Topic Model

#### Example 03 - Hidden Markov Model

#### Example 04 - Bayesian Linear Regression